In [1]:
import os
import json
import numpy as np
import pandas as pd
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
# definindo os caminhos
data_path = os.getenv("DATA_PATH")
results_path = os.getenv("RESULTS_PATH")

In [4]:
# carregando o dataset de valuations processado
df = pd.read_parquet(os.path.join(data_path, "model1_dataset.parquet"))
df.head()

,player_id,date,market_value_in_eur,height_in_cm,international_caps,international_goals,national_team_ranking_inv,position_rank,sub_pos_Attacking Midfield,sub_pos_Central Midfield,...,assists_per_90,minutes_per_game,squad_size,net_transfer_record,national_team_players,stadium_seats,league_tier,club_computed_market_value,year,log_market_value
0,6893,2003-12-15,900000,188.0,0.0,0.0,0,2,False,False,...,0.0,0.0,25,-1350000.0,4,26850,1,0.0,2003,13.710151
1,14007,2004-10-04,750000,177.0,0.0,0.0,0,3,False,False,...,0.0,0.0,26,900000.0,8,18360,1,0.0,2004,13.527830
2,13957,2004-10-04,2000000,182.0,0.0,0.0,0,1,False,False,...,0.0,0.0,0,-0.0,0,5441,4,0.0,2004,14.508658
3,13952,2004-10-04,2000000,175.0,0.0,0.0,0,4,False,False,...,0.0,0.0,27,-24490000.0,12,50033,4,0.0,2004,14.508658
4,13894,2004-10-04,500000,193.0,0.0,0.0,0,1,False,False,...,0.0,0.0,26,-0.0,2,8432,4,0.0,2004,13.122365


In [5]:
# fazendo split dos dados: treino até 2021, validação 2022 e 2023, teste 2024 pra frente
df_train = df[df["date"] < "2022-01-01"].copy()
df_val = df[(df["date"] >= "2022-01-01") & (df["date"] < "2024-01-01")].copy()
df_test = df[df["date"] >= "2024-01-01"].copy()

In [6]:
# mediana de market_value por (position_rank, league_tier) calculada só no treino
lookup = (
    df_train.groupby(["position_rank", "league_tier"])["market_value_in_eur"]
    .median()
    .rename("pred_baseline")
    .reset_index()
)

lookup

,position_rank,league_tier,pred_baseline
0,0,2,800000.0
1,0,3,1000000.0
2,0,4,900000.0
3,1,0,750000.0
4,1,1,2275000.0
5,1,2,1500000.0
6,1,3,1500000.0
7,1,4,1000000.0
8,1,5,900000.0
9,2,0,750000.0


In [7]:
# fallbacks
fallback_pos = (
    df_train.groupby("position_rank")["market_value_in_eur"].median().to_dict()
)
fallback_global = df_train["market_value_in_eur"].median()

fallback_pos, fallback_global

({0: 900000.0, 1: 1000000.0, 2: 1300000.0, 3: 1500000.0, 4: 1500000.0},
 1500000.0)

In [8]:
# aplicando o baseline na validação com fallback
lookup_dict = lookup.set_index(["position_rank", "league_tier"])[
    "pred_baseline"
].to_dict()


def predict_baseline(row):
    key = (row["position_rank"], row["league_tier"])
    if key in lookup_dict:
        return lookup_dict[key]
    if row["position_rank"] in fallback_pos:
        return fallback_pos[row["position_rank"]]
    return fallback_global


df_val["pred_baseline"] = df_val.apply(predict_baseline, axis=1)
df_val[["market_value_in_eur", "pred_baseline"]].head()

,market_value_in_eur,pred_baseline
220582,550000,750000.0
220583,3000000,850000.0
220584,500000,750000.0
220585,500000,750000.0
220586,7000000,850000.0


In [9]:
# valor real e valores preditos pela baseline
y_real = df_val["market_value_in_eur"].values
y_pred = df_val["pred_baseline"].values

# calculando métricas do baseline na validação
mae = np.mean(np.abs(y_real - y_pred))
mape = np.mean(np.abs((y_real - y_pred) / (y_real + 1e-6))) * 100
rmse = np.sqrt(np.mean((y_real - y_pred) ** 2))

log_real = np.log1p(y_real)
log_pred = np.log1p(y_pred)
ss_res = np.sum((log_real - log_pred) ** 2)
ss_tot = np.sum((log_real - np.mean(log_real)) ** 2)
r2_log = 1 - ss_res / ss_tot

ss_res_real = np.sum((y_real - y_pred) ** 2)
ss_tot_real = np.sum((y_real - np.mean(y_real)) ** 2)
r2_real = 1 - ss_res_real / ss_tot_real

print(f"MAE: € {mae:>12,.0f}")
print(f"MAPE: {mape:>12.2f} %")
print(f"RMSE: € {rmse:>12,.0f}")
print(f"R² (log scale): {r2_log:.4f}")
print(f"R² (real scale): {r2_real:.4f}")

MAE: €    3,409,454
MAPE:        70.07 %
RMSE: €    9,463,828
R² (log scale): 0.1360
R² (real scale): -0.0332


In [10]:
# salvando as métricas do baseline
metrics = {
    "dataset": "validation",
    "split": {"start": "2022-01-01", "end": "2023-12-31"},
    "n_samples": int(len(df_val)),
    "MAE_eur": float(mae),
    "MAPE_pct": float(mape),
    "RMSE_eur": float(rmse),
    "R2_log": float(r2_log),
    "R2_real": float(r2_real),
    "strategy": "mediana por (position_rank, league_tier) com fallback",
}

out_path = os.path.join(results_path, "baseline_model1_metrics.json")
with open(out_path, "w") as f:
    json.dump(metrics, f, indent=2)